<a href="https://colab.research.google.com/github/azcsprof/UCLA-XL161/blob/Final-Project-Starter-Code/XL161_Final_Project_Option_4_Starter_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COM SCI XL161 FINAL PROJECT — OPTION 4
# Probabilistic Rescue Agent
# ============================================================
#
# Core Idea:
#
# This project models an AI rescue system operating in
# a dangerous and uncertain environment.
#
# The agent must:
#
#   • search for survivors
#   • avoid dangerous zones
#   • reason using rules
#   • update beliefs under uncertainty
#   • learn from outcomes
#
# The important question is NOT:
#
#     "Did the agent reach the goal?"
#
# The important question is:
#
#     "How did the agent decide what was safe,
#      dangerous, or worth exploring?"
#
# ============================================================
#
# MODULE CONNECTIONS
#
# Module 1:
#   Agents, environments, actions, goals
#
# Module 2:
#   State-space search
#
# Module 3:
#   Heuristic prioritization
#
# Module 4:
#   Constraint satisfaction
#
# Module 5:
#   Strategic tradeoffs under limited resources
#
# Module 6:
#   Rule-based reasoning
#
# Module 7:
#   Structured environmental representation
#
# Module 8:
#   Probabilistic reasoning and belief updates
#
# Module 9:
#   Learning from experience
#
# ============================================================

import heapq
import random


# ============================================================
# 1. ENVIRONMENT GRID
# ============================================================
#
# S = start
# G = rescue target
# # = blocked area
# . = open space
# ? = uncertain danger zone
#
# The environment is the world the agent must reason about.
#
# ============================================================

grid = [
    ["S", ".", ".", "#", ".", "."],
    [".", "#", ".", "#", "?", "."],
    [".", "#", ".", ".", ".", "."],
    [".", ".", ".", "#", "#", "."],
    ["?", "#", ".", ".", ".", "G"]
]


START = (0, 0)
GOAL = (4, 5)


# ============================================================
# 2. DANGER PROBABILITIES
# ============================================================
#
# Module 8 connection:
#
# The agent does NOT know whether danger exists with certainty.
#
# Instead, it maintains probabilities.
#
# Example:
#
#   (1,4): 0.80
#
# means:
#
#   "There is an estimated 80% chance this area is dangerous."
#
# ============================================================

danger_probability = {
    (1, 4): 0.80,
    (4, 0): 0.35
}


# ============================================================
# 3. LEARNED RISK PENALTIES
# ============================================================
#
# Module 9 connection:
#
# The agent learns from previous experiences.
#
# Dangerous zones that caused problems receive larger penalties
# in future searches.
#
# ============================================================

learned_risk = {
    (1, 4): 0,
    (4, 0): 0
}


# ============================================================
# 4. ENERGY CONSTRAINT
# ============================================================
#
# Module 4 connection:
#
# The agent cannot search forever.
#
# It has limited energy.
#
# This creates resource constraints that affect decisions.
#
# ============================================================

MAX_ENERGY = 18


# ============================================================
# 5. DISPLAY ENVIRONMENT
# ============================================================
#
# Human-readable visualization.
#
# The agent reasons using internal representations,
# not visual understanding.
#
# ============================================================

def print_grid():

    print("\n=== RESCUE ENVIRONMENT ===")

    for row in grid:
        print(" ".join(row))

    print()


# ============================================================
# 6. VALID STATE CHECK
# ============================================================
#
# Module 4 connection:
#
# Constraints eliminate invalid actions.
#
# Invalid states:
#
#   • outside grid
#   • blocked locations
#
# ============================================================

def valid_state(state):

    row, col = state

    if row < 0 or row >= len(grid):
        return False

    if col < 0 or col >= len(grid[0]):
        return False

    if grid[row][col] == "#":
        return False

    return True


# ============================================================
# 7. POSSIBLE ACTIONS
# ============================================================
#
# Module 1 and 2 connection:
#
# The agent explores neighboring future states.
#
# ============================================================

def neighbors(state):

    row, col = state

    possible_moves = [
        (row - 1, col),
        (row + 1, col),
        (row, col - 1),
        (row, col + 1)
    ]

    valid_moves = []

    for move in possible_moves:

        if valid_state(move):
            valid_moves.append(move)

    return valid_moves


# ============================================================
# 8. HEURISTIC FUNCTION
# ============================================================
#
# Module 3 connection:
#
# The heuristic estimates distance to the goal.
#
# This helps prioritize promising rescue paths.
#
# ============================================================

def heuristic(state, goal):

    row, col = state
    goal_row, goal_col = goal

    return abs(row - goal_row) + abs(col - goal_col)


# ============================================================
# 9. RULE-BASED SAFETY REASONING
# ============================================================
#
# Module 6 connection:
#
# The system uses symbolic rules.
#
# Example:
#
# IF danger probability > 0.70
# THEN classify zone as HIGH RISK
#
# ============================================================

def risk_category(state):

    probability = danger_probability.get(state, 0)

    if probability >= 0.70:
        return "HIGH_RISK"

    if probability >= 0.30:
        return "MODERATE_RISK"

    return "LOW_RISK"


# ============================================================
# 10. STRUCTURED ENVIRONMENT REPRESENTATION
# ============================================================
#
# Module 7 connection:
#
# The system represents relationships:
#
#   connected(room_a, room_b)
#   dangerous(zone_1)
#
# instead of isolated facts.
#
# ============================================================

def describe_environment():

    print("\n=== STRUCTURED ENVIRONMENT KNOWLEDGE ===")

    for row in range(len(grid)):

        for col in range(len(grid[0])):

            current = (row, col)

            if valid_state(current):

                for adjacent in neighbors(current):

                    print(f"connected({current}, {adjacent})")

                if current in danger_probability:

                    print(f"dangerous({current})")


# ============================================================
# 11. MOVEMENT COST FUNCTION
# ============================================================
#
# This combines multiple forms of reasoning:
#
#   • movement effort
#   • rule-based danger
#   • probability
#   • learned experience
#
# ============================================================

def movement_cost(state):

    base_cost = 1

    probability = danger_probability.get(state, 0)

    learned_penalty = learned_risk.get(state, 0)

    category = risk_category(state)

    if category == "HIGH_RISK":
        category_penalty = 6

    elif category == "MODERATE_RISK":
        category_penalty = 3

    else:
        category_penalty = 0

    probability_penalty = probability * 4

    total = (
        base_cost
        + category_penalty
        + probability_penalty
        + learned_penalty
    )

    return total


# ============================================================
# 12. BAYESIAN BELIEF UPDATE
# ============================================================
#
# Module 8 connection:
#
# The agent updates danger beliefs after receiving sensor evidence.
#
# Example:
#
#   smoke detected
#
# increases estimated danger probability.
#
# ============================================================

def update_belief(state, smoke_detected):

    old_probability = danger_probability.get(state, 0.20)

    if smoke_detected:

        # Increase danger estimate.
        new_probability = min(1.0, old_probability + 0.20)

    else:

        # Reduce danger estimate.
        new_probability = max(0.0, old_probability - 0.10)

    danger_probability[state] = new_probability

    print(f"\nBelief update for {state}")
    print(f"Old probability: {old_probability:.2f}")
    print(f"New probability: {new_probability:.2f}")


# ============================================================
# 13. A* SEARCH WITH RISK REASONING
# ============================================================
#
# Modules 2–3 integration:
#
# The system searches paths while balancing:
#
#   • distance
#   • uncertainty
#   • learned danger
#
# ============================================================

def rescue_search(start, goal):

    frontier = []

    heapq.heappush(frontier, (0, start))

    came_from = {}
    cost_so_far = {}

    came_from[start] = None
    cost_so_far[start] = 0

    print("\n=== RESCUE SEARCH TRACE ===")

    while frontier:

        current_priority, current = heapq.heappop(frontier)

        print(f"\nExpanding state: {current}")

        if current == goal:

            print("Goal reached.")

            break

        for next_state in neighbors(current):

            step_cost = movement_cost(next_state)

            new_cost = cost_so_far[current] + step_cost

            print(f"  Considering move to {next_state}")
            print(f"    Risk category: {risk_category(next_state)}")
            print(f"    Step cost: {step_cost:.2f}")

            # Constraint:
            # do not exceed energy limit.
            if new_cost > MAX_ENERGY:

                print("    Move rejected: energy constraint exceeded.")

                continue

            if (
                next_state not in cost_so_far or
                new_cost < cost_so_far[next_state]
            ):

                cost_so_far[next_state] = new_cost

                priority = (
                    new_cost
                    + heuristic(next_state, goal)
                )

                heapq.heappush(
                    frontier,
                    (priority, next_state)
                )

                came_from[next_state] = current

                print(f"    Updated best path.")
                print(f"    Priority: {priority:.2f}")

    return came_from, cost_so_far


# ============================================================
# 14. RECONSTRUCT PATH
# ============================================================
#
# After search finishes,
# reconstruct final rescue path.
#
# ============================================================

def reconstruct_path(came_from, start, goal):

    if goal not in came_from:
        return []

    current = goal

    path = []

    while current != start:

        path.append(current)

        current = came_from[current]

    path.append(start)

    path.reverse()

    return path


# ============================================================
# 15. LEARNING FROM EXPERIENCE
# ============================================================
#
# Module 9 connection:
#
# If the agent travels through dangerous areas,
# future penalties increase.
#
# This changes future search behavior.
#
# ============================================================

def update_learning(path):

    print("\n=== LEARNING UPDATE ===")

    for state in path:

        if state in danger_probability:

            if danger_probability[state] >= 0.50:

                learned_risk[state] += 2

                print(f"Negative experience at {state}")

                print(
                    f"New learned penalty: "
                    f"{learned_risk[state]}"
                )


# ============================================================
# 16. SENSOR SIMULATION
# ============================================================
#
# Module 8 connection:
#
# Sensors are noisy.
#
# The system does not observe perfect truth.
#
# ============================================================

def simulate_sensor_reading(state):

    actual_probability = danger_probability.get(state, 0)

    random_value = random.random()

    return random_value < actual_probability


# ============================================================
# 17. FULL AGENT EXECUTION
# ============================================================
#
# This integrates:
#
#   • search
#   • constraints
#   • heuristics
#   • rules
#   • uncertainty
#   • learning
#
# ============================================================

def run_rescue_agent():

    print_grid()

    describe_environment()

    # Simulate environmental sensing.
    for state in danger_probability.keys():

        smoke_detected = simulate_sensor_reading(state)

        update_belief(state, smoke_detected)

    came_from, costs = rescue_search(START, GOAL)

    path = reconstruct_path(came_from, START, GOAL)

    print("\n=== FINAL RESCUE PATH ===")

    if path:

        print(path)

        print(f"Total rescue cost: {costs[GOAL]:.2f}")

    else:

        print("No valid rescue path found.")

    update_learning(path)


# ============================================================
# 18. RUN MULTIPLE TRIALS
# ============================================================
#
# Students should observe:
#
#   • belief updates
#   • changing path choices
#   • learned avoidance behavior
#   • interaction between symbolic and probabilistic reasoning
#
# ============================================================

print("\n================================================")
print("PROBABILISTIC RESCUE AGENT")
print("================================================")

print("\nFIRST TRIAL:")
run_rescue_agent()

print("\n\nSECOND TRIAL AFTER LEARNING:")
run_rescue_agent()

# STUDENT EXTENSIONS
# 1. Add additional hazard zones.
# 2. Add survivor locations with probabilities.
# 3. Add limited battery recharge stations.
# 4. Improve heuristic function.
# 5. Add stronger symbolic rules.
# 6. Change belief update behavior.
# 7. Add multiple rescue goals.
# 8. Break the system intentionally:
#       bad probabilities
#       weak rules
#       incorrect learning updates
# 9. Improve the system afterward.